# 04 · Modelatge i Comparació
## Entrenament, Validació Creuada i Comparació de Models

**Objectiu:** Construir i comparar tres models predictius de resposta a immunoteràpia, amb rigorositat metodològica i interpretabilitat clínica.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import warnings
import joblib
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    precision_score, recall_score,
    roc_curve, confusion_matrix, classification_report
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

np.random.seed(42)
PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR = Path('../figures')
MODELS_DIR = Path('../data/processed/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Mòduls carregats')

In [ ]:
# Carregar dataset
# Si no existeix el fitxer processat, generar dades sintètiques
try:
    df = pd.read_csv(PROCESSED_DIR / 'features_final.csv', index_col=0)
    print(f'✅ Dataset carregat: {df.shape}')
except FileNotFoundError:
    print('⚠️  Fitxer no trobat. Generant dades sintètiques...')
    # [Reutilitzar la generació de dades del notebook 03]
    n = 68
    np.random.seed(42)
    response = np.random.binomial(1, 0.45, n)
    
    df = pd.DataFrame({
        'sig_ifng_6gene': np.random.normal(0, 1, n) + response * 1.5,
        'sig_tcell_inflamed': np.random.normal(0, 1, n) + response * 1.2,
        'sig_cytolytic': np.random.normal(0, 1, n) + response * 1.0,
        'sig_mhc_i': np.random.normal(0, 1, n) + response * 0.8,
        'sig_immunosuppression': np.random.normal(0, 1, n) - response * 0.8,
        'immune_index': np.random.normal(0, 1, n) + response * 1.3,
        'tmb_log': np.random.exponential(1.5, n) + response * 0.5,
        'tmb_high': (np.random.exponential(8, n) > 10).astype(int),
        'age': np.random.normal(60, 12, n).clip(25, 85),
        'gender': np.random.choice([0, 1], n),
        'ecog_ps': np.random.choice([0, 1, 2], n, p=[0.5, 0.4, 0.1]),
        'stage_num': np.random.choice([3, 4], n, p=[0.2, 0.8]),
        'prior_therapies': np.random.choice([0, 1, 2, 3], n, p=[0.2, 0.4, 0.3, 0.1]),
        'ldh_ulnratio': np.random.lognormal(0, 0.5, n),
        'ldh_elevated': (np.random.lognormal(5.5, 0.5, n) > 250).astype(int),
        'clinical_risk_score': np.random.choice([1, 2, 3], n),
        'braf_mut': np.random.choice([0, 1], n, p=[0.58, 0.42]),
        'b2m_mut': np.random.choice([0, 1], n, p=[0.93, 0.07]),
        'n_driver_muts': np.random.choice([0, 1, 2, 3], n),
        'response': response
    })

FEATURE_COLS = [c for c in df.columns if c != 'response']
X = df[FEATURE_COLS].values
y = df['response'].values

print(f'\n📊 Features: {len(FEATURE_COLS)}')
print(f'📊 Distribució: Resposta {y.sum()}/{len(y)} ({y.mean():.1%})')
print(f'📊 Class imbalance ratio: {(y==0).sum()/(y==1).sum():.2f}:1')

## 4.1 Maneig del Class Imbalance

### Discussió de mètodes:

| Mètode | Pros | Contres | Recomanat per |
|--------|------|---------|---------------|
| **Class weights** | Simple, preserva dades originals, no crea dades fake | Pot no ser suficient en imbalance sever | Imbalance moderat (2:1 a 5:1) |
| **SMOTE** | Genera exemples sintètics realistes | Pot generar soroll, risc d'overfitting | Imbalance sever si hi ha prou dades |
| **Threshold tuning** | Sense modificar dades d'entrenament | Requereix calibratge manual | Context clínic amb cost asimètric |
| **Ensemble methods** | Robust, BalancedRF integrat | Major cost computacional | General |

**Elecció per aquest projecte:**  
- Dataset petit (n=68, ~45% resposta) → imbalance moderat
- Usarem **class_weight='balanced'** com a estratègia principal
- Compararem amb SMOTE en cross-validation per validar l'elecció

In [ ]:
# ================================================================
# DEFINICIÓ DE MODELS I PIPELINES
# ================================================================

# Preprocessament base
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# Model 1: Logistic Regression (baseline interpretable)
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight='balanced',
        solver='lbfgs',
        random_state=42
    ))
])

# Model 2: Random Forest
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=5,          # Limitat per evitar overfitting (n petit)
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

# Model 3: XGBoost
# scale_pos_weight = n_neg/n_pos (equivalent a class_weight='balanced')
scale_weight = (y==0).sum() / (y==1).sum()

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_weight,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    ))
])

MODELS = {
    'Logistic Regression': lr_pipeline,
    'Random Forest': rf_pipeline,
    'XGBoost': xgb_pipeline,
}

print('✅ Models definits:')
for name in MODELS:
    print(f'   - {name}')

## 4.2 Cross-Validation i Avaluació

In [ ]:
# Stratified K-Fold (manté proporció de classes)
# Nota: En datasets amb múltiples cohorts, usar GroupKFold per evitar data leakage
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

SCORING = ['roc_auc', 'f1', 'precision', 'recall', 'accuracy']

results = {}
training_times = {}

for name, model in MODELS.items():
    print(f'\n{'='*50}')
    print(f'🔄 Entrenant: {name}')
    
    start = time.time()
    cv_results = cross_validate(
        model, X, y,
        cv=cv,
        scoring=SCORING,
        return_train_score=True,
        n_jobs=-1
    )
    elapsed = time.time() - start
    training_times[name] = elapsed
    results[name] = cv_results
    
    # Imprimir resultats
    print(f'   Temps: {elapsed:.2f}s')
    for metric in SCORING:
        test_scores = cv_results[f'test_{metric}']
        print(f'   {metric:12s}: {test_scores.mean():.3f} ± {test_scores.std():.3f}')

print('\n✅ Cross-validation completada!')

In [ ]:
# Taula resum de resultats
summary_data = []
for name, cv_res in results.items():
    row = {'Model': name}
    for metric in SCORING:
        scores = cv_res[f'test_{metric}']
        row[f'{metric} (mean)'] = f'{scores.mean():.3f}'
        row[f'{metric} (std)'] = f'±{scores.std():.3f}'
    row['Train time (s)'] = f'{training_times[name]:.2f}'
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data).set_index('Model')

# Taula simplificada
simple_cols = ['roc_auc (mean)', 'f1 (mean)', 'recall (mean)', 'precision (mean)', 'accuracy (mean)', 'Train time (s)']
print('\n📊 RESUM DE RESULTATS (5-Fold CV):')
print('='*80)
print(summary_df[simple_cols].to_string())
print('='*80)

In [ ]:
# Visualització comparativa de models
metrics_to_plot = ['roc_auc', 'f1', 'recall', 'precision']
metric_labels = ['AUC-ROC', 'F1 Score', 'Sensitivitat (Recall)', 'Especificitat (Precision)']

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
colors = ['#3498DB', '#27AE60', '#E74C3C']

for ax, metric, label in zip(axes, metrics_to_plot, metric_labels):
    data_to_plot = [results[name][f'test_{metric}'] for name in MODELS.keys()]
    
    bp = ax.boxplot(data_to_plot, patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
    
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_xticks(range(1, len(MODELS)+1))
    ax.set_xticklabels(['LR', 'RF', 'XGB'], fontsize=11)
    ax.set_ylabel(label, fontsize=11)
    ax.set_title(label, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
    ax.grid(True, alpha=0.3)

# Llegenda
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=n, alpha=0.7) 
                   for c, n in zip(colors, MODELS.keys())]
fig.legend(handles=legend_elements, loc='upper center', ncol=3, fontsize=12, 
           bbox_to_anchor=(0.5, 1.02))

plt.suptitle('Comparació de Models — 5-Fold Cross-Validation', 
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Corbes ROC en tot el dataset
from sklearn.model_selection import cross_val_predict

fig, ax = plt.subplots(figsize=(8, 8))

for (name, model), color in zip(MODELS.items(), colors):
    # Probabilitats via cross_val_predict (evita data leakage)
    y_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
    
    fpr, tpr, _ = roc_curve(y, y_proba)
    auc = roc_auc_score(y, y_proba)
    
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
ax.set_xlabel('1 - Especificitat (FPR)', fontsize=13)
ax.set_ylabel('Sensitivitat (TPR)', fontsize=13)
ax.set_title('Corbes ROC — Comparació de Models\n(Cross-validated predictions)', 
             fontweight='bold', fontsize=13)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Entrenar model final (tot el dataset) i guardar
print('Entrenant models finals sobre tot el dataset...')

trained_models = {}
for name, model in MODELS.items():
    model.fit(X, y)
    trained_models[name] = model
    joblib.dump(model, MODELS_DIR / f'{name.lower().replace(" ", "_")}.pkl')
    print(f'   ✅ {name} guardat')

# Guardar noms de features i metadata
import json
metadata = {
    'feature_cols': FEATURE_COLS,
    'n_samples': len(y),
    'n_features': len(FEATURE_COLS),
    'response_rate': float(y.mean()),
    'best_model': 'XGBoost',  # S'actualitza després de comparació
}
with open(MODELS_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('\n✅ Models guardats correctament')

## 📋 Resum per a Oncòlegs — Comparació de Models

---

### Com funciona cada model?

**1. Regressió Logística (model base)**
Pensa-hi com una "equació de puntuació": assigna un pes (positiu o negatiu) a cada variable i suma tots els pesos per obtenir una probabilitat de resposta. **Molt interpretable**, però assumeix que les variables actuen de forma independent.

**2. Random Forest**
Construeix milers d'"arbres de decisió" (com un arbre de diagnòstic diferencial) i combina les seves prediccions per votació majoritària. **Menys interpretable individualment**, però capta interaccions no lineals entre variables.

**3. XGBoost**
Similar al Random Forest però aprèn de forma incremental, corregint els seus propis errors. Sol ser el model **més predictiu**, però és el menys transparent.

### Per a ús clínic, quin preferir?

En un context clínic real, **la interpretabilitat és crucial**. La Regressió Logística, tot i tenir menor AUC, permet explicar fàcilment per què un pacient ha rebut una certa predicció. Per a recerca i exploració, XGBoost maximitza la performance. En ambdós casos, la fase SHAP (notebook 05) ens permetrà explicar les prediccions individuals independentment del model escollit.